# TP 2 — Comparer ML et deep learning sur Tiny Shakespeare

Je compare machine learning et deep learning sur l'identification du locuteur dans Tiny Shakespeare. Je pars du TP1 et de ses analyses pour construire un pipeline de deep learning. Le protocole d'évaluation ne change pas : les réseaux changent la représentation et le modèle, pas la façon de mesurer.

| Étape | Section | Ce que je livre |
|---|---|---|
| 1. Nettoyage et prétraitement | section 1 | Les 962 répliques du TP1, le découpage, un tokenizer commun aux modèles ML et DL. |
| 2. Représentation | section 2 | La référence TF–IDF du TP1 et des embeddings appris avec le classifieur. |
| 3. Comparaison ML / DL | section 3 | Classe majoritaire, LinearSVC, puis MLP, LSTM et GRU sur les mêmes cinq folds. |
| 4. Évaluation | section 4 | Scores par fold, erreurs relues réplique par réplique, expériences ciblées, test final, journal, conclusion. |


## 0. Mon problème et les acquis du TP1

Je prédis le locuteur d'une réplique à partir de son seul texte, parmi 10 classes. Je garde les répliques d'au moins 10 mots des 10 locuteurs les plus fréquents, comme au TP1.

| Fait observé dans le TP1 | Ce que j'examine dans le TP2 |
|---|---|
| 962 répliques, de 78 à 129 par classe | Le sur-apprentissage sur un petit jeu modérément déséquilibré. |
| 9 confusions JULIET–ROMEO (4 + 5) et 7 de GLOUCESTER vers DUKE VINCENTIO sur le test | Les indices communs aux erreurs, dont la pièce d'origine. |
| `kate` (2,645), `romeo` (1,623), `paulina` (1,447), `camillo` (1,211) : poids les plus élevés (TP1, section 8.3) | Indices liés aux personnages, donc à la pièce, ou reconnaissance du style ? |
| LinearSVC : F1 macro 0,482 ± 0,035 en validation croisée, 1 sur le train des folds, 0,563 sur le test | La généralisation, pas l'ajustement au train ; 0,563 sert de repère dans la section 4.4. |

> *Mes hypothèses :*
>
> - **H1, indices lexicaux.** Un MLP sur la moyenne des embeddings reste à moins de 0,05 de F1 macro de TF–IDF + LinearSVC : les deux ignorent l'ordre. Testée dans la section 3.4.
> - **H2, ordre des mots.** LSTM et GRU ne dépassent le MLP que si l'ordre porte un signal ; mélanger les mots doit alors les dégrader, pas le MLP. Testée dans les sections 3.5, 3.6 et 4.3. Les bigrammes du TF–IDF captent déjà un peu d'ordre local.
> - **H3, mémorisation.** Sans régularisation, un réseau atteint un F1 macro train proche de 1 en moins de 20 epochs, avec un écart train/validation supérieur à 0,3. Dropout, weight decay et arrêt sur la courbe de validation réduisent l'écart sans forcément améliorer la validation. Testée par les courbes des sections 3.4 à 3.6.
> - **H4, coût.** À largeur égale, le GRU a moins de paramètres et s'entraîne plus vite que le LSTM, pour un F1 macro comparable. Testée dans les sections 3.6 et 4.1.

Je décide sur les scores moyens, leur dispersion et le coût, avant de regarder le test.


### Mon environnement

Je sélectionne `.venv/bin/python` comme kernel. J'installe les dépendances dans le terminal :

```sh
uv pip install --python .venv/bin/python -r requirements.txt
```

> *Mes choix :*
>
> - PyTorch, pour écrire moi-même forward, loss, backward et mise à jour des poids.
> - GPU Apple par MPS, le backend GPU de PyTorch sur Mac, si disponible, sinon CPU. Un M3 Max suffit pour des réseaux de quelques centaines de milliers de paramètres sur 962 répliques. Les versions sont figées dans `requirements.txt`.
> - Seed 42, la graine du générateur aléatoire, réinitialisée avant chaque essai par une fonction appelée à chaque fold (section 3.1). MPS n'est pas déterministe : je mesure la variabilité par la dispersion entre folds et je vérifie deux runs à seed égale. En cas d'écart, je repasse sur CPU, aussi rapide à cette taille.


In [41]:
# Environnement : versions, matériel, device PyTorch et seed unique pour random, numpy et torch.
import platform
import random
import subprocess
import sys
from importlib.metadata import version

import numpy as np
import torch

# Seed fixée ici ; la fonction de la section 3.1 la réinitialisera avant chaque entraînement.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# GPU Apple par MPS si disponible, sinon CPU
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Versions et dépendances
print(f'Python : {platform.python_version()} — {sys.executable}')
for paquet in ('numpy', 'pandas', 'matplotlib', 'scikit-learn', 'ipykernel', 'torch'):
    print(f'{paquet} : {version(paquet)}')

# Système et processeur
print(f'Système : {platform.platform()} — {platform.machine()}')
if sys.platform == 'darwin':
    processeur, ram = subprocess.check_output(
        ['sysctl', '-n', 'machdep.cpu.brand_string', 'hw.memsize'], text=True,
    ).splitlines()
    print(f'Processeur : {processeur} — RAM : {int(ram) / 2**30:g} Gio')

print(f'Device : {device} — Seed : {RANDOM_STATE}')


Python : 3.14.6 — /Users/hopy/Documents/ING3_GIA/ING3_IA_GB_S1/NLP/.venv/bin/python
numpy : 2.5.3
pandas : 3.0.5
matplotlib : 3.11.1
scikit-learn : 1.9.0
ipykernel : 7.3.0
torch : 2.14.0
Système : macOS-26.5-arm64-arm-64bit-Mach-O — arm64
Processeur : Apple M3 Max — RAM : 36 Gio
Device : mps — Seed : 42


## 1. Nettoyage et prétraitement

### 1.1 Retrouver les mêmes répliques

Je reprends le chargement et le parser du TP1 sans dépendre de son état d'exécution. Le nom en tête de bloc donne l'étiquette et ne reste pas dans l'entrée du modèle.

Le parser est mon seul nettoyage. Je garde ponctuation et majuscules : le tokenizer de la section 1.3 décide, mesure à l'appui, ce que les modèles voient.


In [42]:
# Corpus : téléchargement de Tiny Shakespeare et parser du TP1 (bloc = étiquette + réplique).
import re
import urllib.request

import pandas as pd
from IPython.display import display

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Téléchargement du texte brut
with urllib.request.urlopen(URL, timeout=30) as reponse:
    texte = reponse.read().decode('utf-8')


def parser_repliques(txt: str) -> pd.DataFrame:
    """Reprend les règles du TP1 : un bloc, une étiquette et sa réplique."""
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])  # Nom en majuscules puis « : »
        if entete and len(parties) > 1:
            lignes.append({
                'locuteur': entete.group(1).strip(),
                'replique': " ".join(p.strip() for p in parties[1:]).strip(),
            })
    return pd.DataFrame(lignes)


df = parser_repliques(texte)


In [43]:
# Jeu retenu : répliques d'au moins 10 mots, puis les 10 locuteurs les plus fréquents, comme au TP1.
seuil, nb_locuteurs = 10, 10

# Le seuil de longueur s'applique avant de compter les locuteurs.
longueurs = df['replique'].str.split().str.len()
locuteurs = df.loc[longueurs >= seuil, 'locuteur'].value_counts().head(nb_locuteurs).index

donnees = df.loc[
    (longueurs >= seuil) & df['locuteur'].isin(locuteurs),
    ['replique', 'locuteur'],
].copy()  # Ordre et indices du TP1 conservés.


In [44]:
# Contrôles du jeu retenu : effectifs, longueurs, valeurs manquantes, doublons.
effectifs = donnees['locuteur'].value_counts()
longueurs_retenues = donnees['replique'].str.split().str.len()

# Comptes globaux
print(f'Extraction : {len(df):,} répliques, {df["locuteur"].nunique()} labels')
print(f'Jeu retenu : {len(donnees)} répliques, {len(effectifs)} classes')
print(f'Longueurs (mots) : min {longueurs_retenues.min()}, '
      f'médiane {longueurs_retenues.median():g}, max {longueurs_retenues.max()}')
print(f'Valeurs manquantes : {donnees.isna().sum().sum()}')
print(f'Répliques vides : {donnees["replique"].str.strip().eq("").sum()}')
print(f'Textes dupliqués dans le jeu retenu : {donnees["replique"].duplicated().sum()}')

# Effectifs par classe et aperçu
display(effectifs.rename('Répliques').to_frame())
display(donnees.head(3))


Extraction : 7,094 répliques, 297 labels
Jeu retenu : 962 répliques, 10 classes
Longueurs (mots) : min 10, médiane 31, max 578
Valeurs manquantes : 0
Répliques vides : 0
Textes dupliqués dans le jeu retenu : 0


,Répliques
locuteur,
GLOUCESTER,129
DUKE VINCENTIO,125
MENENIUS,107
PETRUCHIO,95
ROMEO,94
CORIOLANUS,87
JULIET,85
LEONTES,82
ISABELLA,80


,replique,locuteur
22,"What work's, my countrymen, in hand? where go ...",MENENIUS
24,"Why, masters, my good friends, mine honest nei...",MENENIUS
26,"I tell you, friends, most charitable care Have...",MENENIUS


> *Mes observations :*
>
> Je retrouve les 962 répliques du TP1 : 7 094 extraites, 297 étiquettes, 10 classes de 78 à 129 répliques après le seuil de 10 mots. Aucune valeur manquante, aucune réplique vide, aucun doublon.
>
> Les longueurs vont de 10 à 578 mots, médiane 31. Cette asymétrie fixe un réglage de la section 2.2 : longueur maximale sur un quantile du train de chaque fold, taux de troncature mesuré. Une longueur au maximum remplirait les batches de padding pour quelques répliques.


### 1.2 Séparer apprentissage, sélection et évaluation

Je reproduis le découpage stratifié du TP1 : 769 répliques de développement, 193 de test, seed 42. Mêmes données, même seed, même version de scikit-learn : le découpage est identique, et la section 1.3 le vérifie avec la taille du vocabulaire TF–IDF du TP1.

Sur les 769 répliques, `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` donne cinq folds communs à tous les modèles.

| Ensemble | Taille | Rôle |
|---|---|---|
| Train du fold | 615 ou 616 | Tout ce qui dépend des données s'y apprend : vocabulaire, IDF, embeddings, poids. |
| Validation du fold | 154 ou 153 | Comparaison des modèles sur les mêmes répliques et choix des réglages, dont le nombre d'epochs. |
| Test | 193 | Une seule évaluation, après avoir figé tous les choix (section 4.4). |

Pas de validation interne pour les réseaux. Une epoch est un passage complet sur le train du fold. Le nombre d'epochs devient un hyperparamètre choisi par validation croisée, comme `C` au TP1 (section 9.4). Chaque réseau s'entraîne un nombre maximal d'epochs dans chaque fold. Je retiens l'epoch où la courbe moyenne de validation des cinq folds est la plus haute : c'est l'epoch retenue (section 3.1). Une validation interne de 123 répliques, 10 à 16 par classe, aurait rendu ce choix bruité. Elle aurait aussi retiré 20 % des données aux réseaux, alors que LinearSVC aurait gardé les 615.


In [45]:
# Découpage développement / test du TP1 : 80/20 stratifié, seed 42.
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    donnees['replique'], donnees['locuteur'],
    test_size=0.2, stratify=donnees['locuteur'], random_state=RANDOM_STATE,
)

# Vérifications : aucun indice ni texte partagé entre développement et test.
assert set(X_train.index).isdisjoint(X_test.index)
assert not set(X_train).intersection(X_test)

# Tailles, empreinte du découpage et répartition des classes
print(f'Développement : {len(X_train)} répliques ; test : {len(X_test)}')
print(f'Premiers indices du test : {list(X_test.index[:10])}')
display(pd.DataFrame({
    'Développement': y_train.value_counts(),
    'Test': y_test.value_counts(),
}).sort_values('Développement', ascending=False))


Développement : 769 répliques ; test : 193
Premiers indices du test : [3009, 6783, 5903, 1321, 3338, 2294, 3344, 2875, 3378, 2212]


,Développement,Test
locuteur,,
GLOUCESTER,103,26
DUKE VINCENTIO,100,25
MENENIUS,85,22
PETRUCHIO,76,19
ROMEO,75,19
CORIOLANUS,70,17
JULIET,68,17
LEONTES,66,16
ISABELLA,64,16


In [46]:
# Cinq folds stratifiés sur les 769 répliques de développement, communs aux modèles ML et DL.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Indices d'origine, utilisables avec .loc, plutôt que positions
folds = [
    {'train': X_train.index[positions_train], 'validation': X_train.index[positions_validation]}
    for positions_train, positions_validation in cv.split(X_train, y_train)
]

# Vérification : les dix classes sont présentes dans chaque validation.
assert all(y_train.loc[fold['validation']].nunique() == y_train.nunique() for fold in folds)

display(pd.DataFrame(
    [{nom: len(indices) for nom, indices in fold.items()} for fold in folds],
    index=pd.RangeIndex(1, len(folds) + 1, name='Fold'),
))


,train,validation
Fold,,
1,615,154
2,615,154
3,615,154
4,615,154
5,616,153


> *Mon analyse :*
>
> 769 répliques de développement et 193 de test, comme au TP1. Les deux vérifications `assert` passent : aucun indice ni texte partagé. La section 1.3 ajoute une preuve : le TF–IDF du TP1 recalculé sur ces 769 répliques retrouve ses 5 915 termes.
>
> Cinq folds communs : 615 répliques d'apprentissage et 154 de validation pour les folds 1 à 4, 616 et 153 pour le dernier. Chaque réplique de développement est évaluée une fois ; les dix classes sont présentes dans chaque validation. La validation sert à comparer les modèles et à choisir les réglages ; le test sert une fois, dans la section 4.4.


### 1.3 Tokeniser : ce que les modèles voient

Je tokenise en mots. Avec 962 répliques, un vocabulaire de quelques milliers de tokens distincts reste apprenable et les séquences restent courtes, médiane 31 mots. Les sous-mots demandent d'apprendre un découpage, hors de ce TP ; les caractères, je les compare ci-dessous en longueur de séquence.

Le TP1 utilisait le `token_pattern` par défaut de scikit-learn, `(?u)\b\w\w+\b` : il supprime les tokens d'une lettre et coupe aux apostrophes. « I'll » devient `ll`, « I » disparaît ; les IDF les plus élevés du TP1 le montraient déjà avec `ll bring` ou `ll go`. Or « I », « not » ou « Romeo » peuvent être des indices de locuteur, et un nettoyage naïf peut détruire le signal cherché.

Je définis donc un tokenizer unique pour TF–IDF et pour les réseaux : la comparaison ML/DL ne porte que sur la représentation et le modèle. Je garde aussi la référence exacte du TP1 dans la section 3.3 ; l'écart entre les deux TF–IDF mesurera l'effet du tokenizer seul. Avant de fixer les règles, je compte les cas rencontrés dans les 769 répliques de développement.


In [47]:
# Faits sur le texte brut des 769 répliques de développement, avant toute décision de découpage.
from collections import Counter

# Chaque cas est un motif regex ; « mot » = lettres et apostrophes, sans découpage imposé.
CAS = {
    'Apostrophe en tête': r"(?<![A-Za-z])'[A-Za-z]+",
    'Apostrophe interne': r"[A-Za-z]+'[A-Za-z]+",
    'Apostrophe en fin': r"[A-Za-z]+'(?![A-Za-z])",
    'Mot d’une lettre': r"(?<![A-Za-z'])[A-Za-z](?![A-Za-z'])",
    'Tiret': r"[A-Za-z]+-[A-Za-z]+",
    'Chiffre': r"\d+",
    'Ponctuation': r"[,;:!?.]",
}


def compter_cas(motif: str) -> dict:
    """Répliques contenant le motif et les trois formes les plus fréquentes qu'il capture."""
    expression = re.compile(motif)
    formes = Counter(forme.lower() for texte in X_train for forme in expression.findall(texte))
    return {
        'Répliques concernées': sum(bool(expression.search(texte)) for texte in X_train),
        'Formes les plus fréquentes': ', '.join(
            f'{forme} ({n})' for forme, n in formes.most_common(3)),
    }


# Un tableau, une ligne par cas
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame({cas: compter_cas(motif) for cas, motif in CAS.items()}).T
            .rename_axis('Cas dans le texte brut'))

# Longueurs médianes en mots et en caractères ; leur rapport chiffre un découpage en caractères.
mots_par_replique = X_train.str.split().str.len()
caracteres_par_replique = X_train.str.len()
caracteres_par_mot = caracteres_par_replique.median() / mots_par_replique.median()
print(f'Longueur médiane : {mots_par_replique.median():g} mots '
      f'et {caracteres_par_replique.median():g} caractères par réplique, '
      f'soit {caracteres_par_mot:.1f} caractères par mot.')


,Répliques concernées,Formes les plus fréquentes
Cas dans le texte brut,,
Apostrophe en tête,119,"'tis (61), 'twere (10), 'twas (8)"
Apostrophe interne,392,"i'll (103), that's (20), ne'er (13)"
Apostrophe en fin,81,"i' (23), o' (22), banished' (4)"
Mot d’une lettre,549,"i (902), a (574), o (128)"
Tiret,149,"to-morrow (15), to-night (7), to-day (7)"
Chiffre,3,3 (3)
Ponctuation,769,", (3783), . (1247), ; (730)"


Longueur médiane : 31 mots et 159 caractères par réplique, soit 5.1 caractères par mot.


In [48]:
# Tokenizer commun aux modèles ML et DL, et reproduction du découpage par défaut du TP1.
MOTIF_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")  # Lettres et apostrophes internes.
MOTIF_DEFAUT = re.compile(r"(?u)\b\w\w+\b")  # token_pattern par défaut de scikit-learn (TP1).


def tokeniser(texte: str) -> list[str]:
    """Tokenizer commun : minuscules, mots et contractions."""
    return MOTIF_TOKEN.findall(texte.lower())


def tokeniser_defaut(texte: str) -> list[str]:
    """Découpage du TP1, pour mesurer ce que je change."""
    return MOTIF_DEFAUT.findall(texte.lower())


def montrer_decoupages(indices: tuple[int, ...]) -> None:
    """Affiche des répliques de développement et leurs deux découpages."""
    for indice in indices:
        print(f'{y_train.loc[indice]} : {X_train.loc[indice]}')
        print(f'   TP1    : {tokeniser_defaut(X_train.loc[indice])}')
        print(f'   commun : {tokeniser(X_train.loc[indice])}\n')


# Trois répliques : contraction, apostrophe en tête, nom propre et mots d'une lettre.
montrer_decoupages((3062, 563, 6253))


ROMEO : Switch and spurs, switch and spurs; or I'll cry a match.
   TP1    : ['switch', 'and', 'spurs', 'switch', 'and', 'spurs', 'or', 'll', 'cry', 'match']
   commun : ['switch', 'and', 'spurs', 'switch', 'and', 'spurs', 'or', "i'll", 'cry', 'a', 'match']

MENENIUS : For 'tis a sore upon us, You cannot tent yourself: be gone, beseech you.
   TP1    : ['for', 'tis', 'sore', 'upon', 'us', 'you', 'cannot', 'tent', 'yourself', 'be', 'gone', 'beseech', 'you']
   commun : ['for', 'tis', 'a', 'sore', 'upon', 'us', 'you', 'cannot', 'tent', 'yourself', 'be', 'gone', 'beseech', 'you']

PETRUCHIO : Nay, hear you, Kate: in sooth you scape not so.
   TP1    : ['nay', 'hear', 'you', 'kate', 'in', 'sooth', 'you', 'scape', 'not', 'so']
   commun : ['nay', 'hear', 'you', 'kate', 'in', 'sooth', 'you', 'scape', 'not', 'so']



In [49]:
# Ce que chaque découpage voit sur les 769 répliques de développement, puis l'empreinte du TP1.
from sklearn.feature_extraction.text import TfidfVectorizer

TOKENISEURS = {'TP1': tokeniser_defaut, 'commun': tokeniser}
PARAMS_TFIDF = dict(ngram_range=(1, 2), min_df=2, max_df=1.0, stop_words=None, sublinear_tf=True)


def compter_tokens(decouper) -> Counter:
    """Fréquence de chaque token sur le développement : une description, pas un apprentissage."""
    return Counter(token for texte in X_train for token in decouper(texte))


def plus_frequents(comptes: dict, n: int = 6) -> str:
    """Les n tokens les plus fréquents d'un dictionnaire de comptes, avec leur compte."""
    tries = sorted(comptes.items(), key=lambda paire: -paire[1])
    return ', '.join(f'{token} ({compte})' for token, compte in tries[:n])


def comparer_decoupages() -> pd.DataFrame:
    """Tokens vus par chaque découpage, et ceux que seul l'un des deux produit."""
    comptes = {nom: compter_tokens(decouper) for nom, decouper in TOKENISEURS.items()}
    seulement = {
        nom: {token: n for token, n in comptes[nom].items() if token not in comptes[autre]}
        for nom, autre in (('TP1', 'commun'), ('commun', 'TP1'))
    }
    return pd.DataFrame({
        'Tokens distincts': {nom: len(compte) for nom, compte in comptes.items()},
        'Tokens': {nom: sum(compte.values()) for nom, compte in comptes.items()},
        'Tokens distincts absents de l’autre': {
            nom: len(absents) for nom, absents in seulement.items()},
        'Tokens absents de l’autre': {
            nom: sum(absents.values()) for nom, absents in seulement.items()},
        'Les plus fréquents absents de l’autre': {
            nom: plus_frequents(absents) for nom, absents in seulement.items()},
    }).rename_axis('Tokenizer')


def taille_vocabulaire(vectoriseur: TfidfVectorizer) -> int:
    """Nombre de termes appris sur les 769 répliques de développement."""
    return len(vectoriseur.fit(X_train).vocabulary_)


with pd.option_context('display.max_colwidth', None):
    display(comparer_decoupages())

# Empreinte du TP1 : le même TF–IDF sur les mêmes 769 répliques doit retrouver ses 5 915 termes.
termes_tp1 = taille_vocabulaire(TfidfVectorizer(lowercase=True, **PARAMS_TFIDF))
termes_commun = taille_vocabulaire(
    TfidfVectorizer(tokenizer=tokeniser, token_pattern=None, lowercase=False, **PARAMS_TFIDF)
)
print(f'Termes TF–IDF (1-2 grammes, min_df=2) : TP1 {termes_tp1:,} ; '
      f'tokenizer commun {termes_commun:,}')


,Tokens distincts,Tokens,Tokens distincts absents de l’autre,Tokens absents de l’autre,Les plus fréquents absents de l’autre
Tokenizer,,,,,
TP1,5139,36091,129,371,"ll (119), st (56), er (26), ne (13), ld (8), rt (4)"
commun,5449,37652,439,2559,"i (927), a (576), o (151), i'll (103), that's (20), ne'er (13)"


Termes TF–IDF (1-2 grammes, min_df=2) : TP1 5,915 ; tokenizer commun 6,108


> *Mes observations :*
>
> Dans le texte brut, l'apostrophe interne touche 392 répliques : « i'll » (103), « that's » (20), « ne'er » (13). L'apostrophe en tête touche 119 répliques, « 'tis » (61) en premier ; l'apostrophe en fin, 81 répliques, « i' » (23) et « o' » (22). Les mots d'une lettre apparaissent dans 549 répliques : « i » (902), « a » (574), « o » (128). Le tiret relie deux mots dans 149 répliques, « to-morrow » (15) en tête ; les chiffres touchent 3 répliques. Une réplique médiane compte 31 mots pour 159 caractères, 5,1 caractères par mot.
>
> Sur les 769 répliques, le tokenizer commun produit 37 652 tokens, dont 5 449 distincts. Le découpage du TP1 produit 36 091 tokens, dont 5 139 distincts. Les 439 tokens distincts propres au tokenizer commun sont d'abord `i` (927 après découpage), `a` (576), `o` (151) et `i'll` (103). Les 129 propres au TP1 sont des fragments : `ll` (119), `st` (56), `er` (26). Le TF–IDF du TP1 recalculé ici retrouve ses 5 915 termes : mêmes 769 répliques. Avec le tokenizer commun, 6 108 termes.

Mes règles, fixées sur ces comptes :

| Cas dans le texte | Décision | Pourquoi |
|---|---|---|
| Majuscules (« Kate », « Romeo ») | minuscules | Même indice en début et en milieu de phrase ; le TP1 faisait de même. |
| Ponctuation | supprimée | Présente dans les 769 répliques ; un séparateur, aucune de mes hypothèses ne porte sur elle. |
| Mots d'une lettre (« I », « a », « O ») | conservés | 549 répliques, « i » 902 fois : pronom, article, interjection. |
| Apostrophe interne (« I'll », « that's », « ne'er ») | un seul token | 392 répliques ; couper « I'll » (103) donne le fragment `ll`. |
| Apostrophe en tête ou en fin (« 'tis », « i' ») | retirée | « 'tis » (61) devient `tis`, « i' » (23) devient `i` : je garde le mot. |
| Tiret (« to-morrow ») | sépare | 149 répliques ; `to` et `morrow` restent lisibles. |
| Chiffres | supprimés | 3 répliques. |
| Caractères comme unités | non | 5,1 caractères par mot : des séquences cinq fois plus longues pour un petit réseau récurrent. |

> *Ma conclusion :* le découpage du TP1 effaçait la première personne et les contractions, 2 559 tokens sur 36 091, soit 7 % du texte. Le tokenizer commun les rend visibles aux deux familles de modèles ; la section 3.3 mesurera si LinearSVC en profite, première ligne du journal.


## 2. Représentation

### 2.1 Ma référence TF–IDF

Je garde les paramètres du TP1, rassemblés dans `PARAMS_TFIDF` dans la section 1.3 : `ngram_range=(1, 2)`, `min_df=2`, `max_df=1.0`, `stop_words=None`, `sublinear_tf=True`. Deux références : le TF–IDF exact du TP1, `token_pattern` par défaut et `lowercase=True`, et le TF–IDF sur le tokenizer commun, `tokenizer=tokeniser, token_pattern=None`.

Le `Pipeline` réapprend vocabulaire et IDF sur le train de chaque fold, sans réutiliser la matrice du TP1. Pour chaque fold, je mesure la taille du vocabulaire, la densité de la matrice et la part des tokens de validation absents du vocabulaire du train : la même mesure que le taux de tokens inconnus des embeddings. Avec `min_df=2` des deux côtés, les unigrammes du TF–IDF commun et le vocabulaire des embeddings doivent coïncider dans chaque fold : je le vérifie dans la section 2.2.


In [50]:
# Outils de la référence TF–IDF : un vectoriseur neuf par fold et les mesures qui décrivent un fold.


def construire_tfidf(reference: str) -> TfidfVectorizer:
    """Vectoriseur neuf : 'TP1' = découpage par défaut, 'commun' = tokenizer de la section 1.3."""
    if reference == 'TP1':
        return TfidfVectorizer(lowercase=True, **PARAMS_TFIDF)
    return TfidfVectorizer(tokenizer=tokeniser, token_pattern=None, lowercase=False, **PARAMS_TFIDF)


def pourcentage(partie: float, total: float) -> float:
    """Part en pourcentage."""
    return 100 * partie / total


def vus_une_fois(detail: dict) -> int:
    """Tokens inconnus présents dans une seule réplique du train, donc exclus par min_df=2."""
    return sum(detail['documents_train'][token] == 1 for token in detail['inconnus'])


def detailler_tokens(
    vectoriseur: TfidfVectorizer, decouper, textes_train: pd.Series, textes_validation: pd.Series,
) -> dict:
    """Unigrammes appris, tokens de validation, inconnus et fréquence documentaire du train."""
    unigrammes = {terme for terme in vectoriseur.vocabulary_ if ' ' not in terme}
    tokens_validation = [token for texte in textes_validation for token in decouper(texte)]
    inconnus = [token for token in tokens_validation if token not in unigrammes]  # Absents du train
    documents_train = Counter(token for texte in textes_train for token in set(decouper(texte)))
    return {'unigrammes': unigrammes, 'tokens_validation': tokens_validation,
            'inconnus': inconnus, 'documents_train': documents_train}


def decrire_reference(
    numero: int, reference: str, textes_train: pd.Series, textes_validation: pd.Series,
) -> tuple[dict, dict]:
    """Ajuste une référence sur le train d'un fold : ligne de bilan et détail des tokens."""
    vectoriseur = construire_tfidf(reference)
    matrice_train = vectoriseur.fit_transform(textes_train)  # Appris sur le train seul.
    matrice_validation = vectoriseur.transform(textes_validation)
    assert matrice_train.shape == (len(textes_train), len(vectoriseur.vocabulary_))
    assert matrice_validation.shape == (len(textes_validation), len(vectoriseur.vocabulary_))

    detail = detailler_tokens(vectoriseur, TOKENISEURS[reference], textes_train, textes_validation)
    taille_matrice = matrice_train.shape[0] * matrice_train.shape[1]
    bilan = {
        'Fold': numero, 'Référence': reference,
        'Termes': len(vectoriseur.vocabulary_), 'Unigrammes': len(detail['unigrammes']),
        'Bigrammes': len(vectoriseur.vocabulary_) - len(detail['unigrammes']),
        'Non nuls (%)': pourcentage(matrice_train.nnz, taille_matrice),
        'Tokens de validation inconnus (%)': pourcentage(
            len(detail['inconnus']), len(detail['tokens_validation'])),
    }
    return bilan, detail


def analyser_inconnus(numero: int, textes_train: pd.Series, details: dict) -> dict:
    """Origine des inconnus : apostrophes, et tokens vus une seule fois dans le train."""
    tp1, commun = details['TP1'], details['commun']
    sans_apostrophe = [token for token in commun['tokens_validation'] if "'" not in token]
    inconnus_sans_apostrophe = sum(token not in commun['unigrammes'] for token in sans_apostrophe)
    propres_au_commun = commun['unigrammes'] - tp1['unigrammes']
    comptes_train = Counter(token for texte in textes_train for token in tokeniser(texte))
    return {
        'Fold': numero,
        'Inconnus TP1': len(tp1['inconnus']),
        'Inconnus commun': len(commun['inconnus']),
        'Commun avec apostrophe (%)': pourcentage(
            sum("'" in token for token in commun['inconnus']), len(commun['inconnus'])),
        'Commun sans tokens à apostrophe (%)': pourcentage(
            inconnus_sans_apostrophe, len(sans_apostrophe)),
        'Vus une fois dans le train, TP1 (%)': pourcentage(vus_une_fois(tp1), len(tp1['inconnus'])),
        'Vus une fois dans le train, commun (%)': pourcentage(
            vus_une_fois(commun), len(commun['inconnus'])),
        'Unigrammes propres au commun': plus_frequents(
            {token: comptes_train[token] for token in propres_au_commun}, n=5),
    }


In [51]:
# Mesures par fold : bilan des deux références, puis origine des tokens inconnus.
bilan_tfidf, detail_inconnus = [], []
unigrammes_tfidf = {}  # (fold, référence) -> unigrammes du train, pour la section 2.2

for numero, fold in enumerate(folds, start=1):
    textes_train, textes_validation = X_train.loc[fold['train']], X_train.loc[fold['validation']]
    details = {}
    for reference in TOKENISEURS:
        bilan, details[reference] = decrire_reference(
            numero, reference, textes_train, textes_validation)
        bilan_tfidf.append(bilan)
        unigrammes_tfidf[(numero, reference)] = details[reference]['unigrammes']
    detail_inconnus.append(analyser_inconnus(numero, textes_train, details))

# Affichages : bilan par fold, moyenne par référence, origine des inconnus
bilan_tfidf = pd.DataFrame(bilan_tfidf).set_index(['Fold', 'Référence'])
display(bilan_tfidf.round(2))
display(bilan_tfidf.groupby('Référence').mean().round(2).rename_axis('Moyenne des cinq folds'))
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(detail_inconnus).set_index('Fold').round(2))


Termes  Unigrammes  Bigrammes  Non nuls (%)  \
Fold Référence                                                
1    TP1          4861        1996       2865          1.00   
     commun       5025        2038       2987          1.02   
2    TP1          4716        1933       2783          1.01   
     commun       4898        1981       2917          1.02   
3    TP1          4688        1922       2766          1.00   
     commun       4843        1968       2875          1.02   
4    TP1          4644        1945       2699          1.00   
     commun       4806        1993       2813          1.01   
5    TP1          4807        1978       2829          1.01   
     commun       4970        2023       2947          1.02   

                Tokens de validation inconnus (%)  
Fold Référence                                     
1    TP1                                    13.67  
     commun                                 14.12  
2    TP1                                    15.34  
     commun                                 15.54  
3    TP1                                    14.64  
     commun                                 14.98  
4    TP1                                    13.88  
     commun                                 14.28  
5    TP1                                    13.70  
     commun                                 14.20

,Termes,Unigrammes,Bigrammes,Non nuls (%),Tokens de validation inconnus (%)
Moyenne des cinq folds,,,,,
TP1,4743.2,1954.8,2788.4,1.00,14.24
commun,4908.4,2000.6,2907.8,1.02,14.62


,Inconnus TP1,Inconnus commun,Commun avec apostrophe (%),Commun sans tokens à apostrophe (%),"Vus une fois dans le train, TP1 (%)","Vus une fois dans le train, commun (%)",Unigrammes propres au commun
Fold,,,,,,,
1,898,969,9.80,13.05,35.52,35.29,"i (747), a (472), o (128), i'll (87), that's (18)"
2,1097,1157,8.04,14.65,37.65,37.17,"i (748), a (468), o (127), i'll (87), that's (16)"
3,1096,1170,8.29,14.07,38.78,36.84,"i (741), a (451), o (111), i'll (83), that's (13)"
4,1110,1191,8.56,13.36,37.48,36.78,"i (728), a (447), o (121), i'll (75), that's (18)"
5,943,1021,9.30,13.20,36.37,36.04,"i (744), a (466), o (117), i'll (80), that's (15)"


> *Mes observations :*
>
> Sur le train d'un fold, 615 répliques, le TF–IDF du TP1 compte de 4 644 à 4 861 termes, contre 5 915 sur les 769 répliques de développement. Moins de répliques, moins de termes qui atteignent `min_df=2`. Le tokenizer commun ajoute de 155 à 182 termes par fold, 4 908 contre 4 743 en moyenne. De 42 à 48 sont des unigrammes, les mêmes dans chaque fold : `i`, `a`, `o`, `i'll`, `that's`. Les bigrammes font près de 60 % des termes dans les deux cas.
>
> La matrice reste creuse, 1,0 % de valeurs non nulles. En validation, 14,2 % des tokens du TP1 et 14,6 % des tokens du tokenizer commun sont absents du vocabulaire du train, de 13,7 % à 15,5 % selon le fold. Entre 8 % et 10 % des inconnus du tokenizer commun contiennent une apostrophe : contractions et possessifs gardés entiers. Sans les tokens à apostrophe, son taux d'inconnus va de 13,1 % à 14,7 %, sous celui du TP1 dans chaque fold. De 35 % à 39 % des tokens inconnus apparaissent une seule fois dans le train ; `min_df=2` les a exclus.
>
> *Ma conclusion :* les deux références ont la même taille et la même densité ; seul le découpage change, ce qui isole l'effet du tokenizer dans la section 3.3. Un token de validation sur sept est inconnu du train. C'est le taux d'inconnus que les embeddings de la section 2.2 devront absorber avec `<UNK>` ; TF–IDF, lui, ignore ces tokens sans coût explicite.


### 2.2 Des tokens aux embeddings

Chaque token reçoit un identifiant, puis une ligne d'une matrice d'embeddings apprise avec le classifieur. Pas de vecteurs pré-entraînés : l'anglais de Shakespeare s'éloigne des corpus modernes, et la comparaison avec TF–IDF reste à données égales.

Un seul vocabulaire par fold pour les trois réseaux, construit sur le train du fold avec la règle de `min_df=2` : un token doit apparaître dans au moins deux répliques. `<UNK>` couvre les tokens absents, `<PAD>` complète les batches. Je mesure le taux de tokens inconnus en validation et le taux de troncature.

La longueur maximale vient d'un quantile des longueurs du train du fold. La dimension des embeddings se justifie par les paramètres qu'elle crée. Environ 2 000 tokens distincts par fold et une dimension de 64 font déjà près de 130 000 paramètres pour 615 répliques : ce rapport paramètres/exemples est la base chiffrée de H3.

La moyenne des embeddings ignore l'ordre : « Romeo loves Juliet » et « Juliet loves Romeo » ont la même moyenne, un LSTM ou un GRU peut les distinguer. J'exclus le padding de la moyenne et je récupère l'état récurrent après le dernier token réel.


In [52]:
# TODO : construire le vocabulaire du train du fold avec le tokenizer commun, les tokens présents
#   dans au moins deux répliques du train, <UNK> et <PAD> ; vérifier son égalité avec
#   unigrammes_tfidf[(fold, 'commun')].
# TODO : fixer la longueur maximale sur un quantile du train du fold ; mesurer inconnus et
#   troncature en validation.
# TODO : préparer le Dataset PyTorch, la fonction d'assemblage des batches, longueurs, masques et un
#   encodage des 10 labels fixé une fois, par ordre alphabétique.
# TODO : justifier la dimension des embeddings par le rapport paramètres/exemples.
# TODO : vérifier qu'ajouter du padding ne change pas la représentation d'une réplique
#   (torch.allclose).


## 3. Comparaison entre modèles ML et DL

### 3.1 Préparer l'entraînement commun

Je compare la classe majoritaire, LinearSVC, puis MLP, LSTM et GRU. Cross-entropy pour entraîner les réseaux, F1 macro pour les comparer : une classe parmi dix, chaque classe au même poids.

Pas de Transformer ici : je pars d'une baseline simple et j'ajoute de la complexité seulement si le gain mesuré la justifie. Avec 615 répliques par fold, un réseau récurrent compact reste le bon niveau de capacité.

Avant les folds, je contrôle un batch : dimensions, dix scores par réplique, loss initiale proche de ln 10 ≈ 2,303 pour dix classes presque équiprobables, gradients finis. Puis un réseau doit sur-apprendre un seul batch jusqu'à une loss proche de 0, et deux runs à seed égale doivent donner les mêmes scores.

| Réglage | Valeur de départ | Pourquoi |
|---|---|---|
| Loss | cross-entropy | Dix classes exclusives. |
| Optimiseur | Adam, taux d'apprentissage 1e-3 | Pas adaptatif par paramètre, robuste sans réglage fin. |
| Batch | 32 répliques | Une vingtaine de mises à jour par epoch sur 615 répliques. |
| Epochs | 40 au maximum ; epoch retenue sur la courbe moyenne de validation | L'epoch est un hyperparamètre choisi par validation croisée, comme `C` au TP1. |
| Dropout, weight decay | 0, puis testés dans la section 4.3 | Je mesure d'abord le sur-apprentissage sans le corriger. |
| Gradient clipping | norme du gradient plafonnée à 1 | Évite les gradients qui explosent dans les séquences longues. |
| Pondération des classes | aucune | Déséquilibre modéré, 78 à 129 ; le F1 macro surveille chaque classe. |
| Seeds | 42, réinitialisée à chaque fold ; 3 seeds si le temps le permet | Séparer la dispersion des folds de celle de l'initialisation. |

Ces réglages tiennent dans un dictionnaire `REGLAGES` ; une expérience de la section 4.3 en change une seule clé. Pour chaque réseau, je trace loss et F1 macro sur train et validation à chaque epoch. L'epoch retenue est la plus petite epoch à moins de 0,5 point du maximum de la courbe moyenne des cinq folds. Le score de validation à cette epoch est un score de sélection, un peu optimiste ; le test de la section 4.4 tranche.


In [53]:
# TODO : définir REGLAGES et une fonction qui réinitialise la seed avant chaque (fold, seed).
# TODO : préparer une boucle commune : forward, cross-entropy, backward et mise à jour des poids.
# TODO : contrôler un batch : dimensions, loss initiale proche de ln 10, gradients finis, un batch
#   sur-appris, deux runs à seed égale.
# TODO : enregistrer loss et F1 macro par epoch sur train et validation ; garder les prédictions de
#   validation par indice.
# TODO : écrire evaluer_ml et evaluer_reseau sur la même boucle de folds et le même scoreur ;
#   compter les paramètres et chronométrer.


### 3.2 Baseline majoritaire

> *Mon hypothèse :* Prédire la classe la plus fréquente du train donne un repère faible mais nécessaire. Un modèle textuel doit apporter un gain mesuré par rapport à ce repère.


In [54]:
# TODO : évaluer DummyClassifier(strategy="most_frequent") sur les cinq folds.
# TODO : conserver les prédictions, l’accuracy et le F1 macro de chaque fold.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.3 TF–IDF + LinearSVC

> *Mon hypothèse :* Sur ce petit jeu, les indices lexicaux restent compétitifs face aux réseaux ; LinearSVC convient aux vecteurs TF–IDF creux. Le tokenizer commun aide un peu, « I » et les contractions devenant visibles, sans changer l'ordre de grandeur.

Je reprends `C=1`, `class_weight=None`, `max_iter=5000`, `random_state=42`, et j'évalue les deux références de la section 2.1 sur les folds communs. Leur écart mesure l'effet du tokenizer seul, première ligne du journal.

Je regarde : F1 macro et accuracy par fold, moyenne et écart-type, écart train/validation, temps.


In [55]:
# TODO : évaluer les deux Pipelines TF–IDF + LinearSVC sur les folds communs.
# TODO : conserver les prédictions par indice, les scores train/validation et les temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.4 Embeddings + moyenne + MLP

> *Mon hypothèse :* Un petit MLP apprend des combinaisons utiles d'indices après la moyenne des embeddings, sans l'ordre des mots. Si H1 tient, son F1 macro reste à moins de 0,05 de LinearSVC.

Une couche cachée ReLU puis dix scores de classe. Je justifierai sa largeur par le nombre de paramètres et le sur-apprentissage observé.

Je regarde : courbes loss et F1 macro train/validation, epoch retenue, écart train/validation à cette epoch, F1 macro par fold, paramètres, temps.


In [56]:
# TODO : construire embeddings, moyenne des embeddings hors padding, couche cachée ReLU et sortie à
#   10 classes.
# TODO : entraîner dans chaque fold avec la boucle commune, 40 epochs au maximum, courbes
#   enregistrées.
# TODO : choisir l'epoch retenue sur la courbe moyenne de validation ; conserver prédictions,
#   courbes et temps.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.5 Embeddings + LSTM

> *Mon hypothèse :* Lire la séquence apporte des indices que la moyenne perd, malgré le faible nombre de répliques. Si H2 tient, le LSTM dépasse le MLP et le mélange des mots de la section 4.3 le fait redescendre.

Une couche unidirectionnelle, puis une couche de sortie linéaire à dix scores. Je garde une version bidirectionnelle pour la section 4.3 : classer une réplique entière n'impose aucune contrainte causale.

Je regarde : les mêmes éléments que pour le MLP, plus la part de répliques tronquées et la norme des gradients.


In [57]:
# TODO : construire embeddings, LSTM à une couche et couche de sortie linéaire à 10 classes.
# TODO : utiliser les longueurs réelles pour exclure le padding de l'état retenu.
# TODO : évaluer avec la boucle, les folds, le budget d'epochs et la règle d'epoch retenue communs.


> *Mes observations :*
>
> *Ma conclusion :*


### 3.6 Embeddings + GRU : comparaison avec le LSTM

> *Mon hypothèse :* Le GRU suffit pour nos répliques avec moins de paramètres qu'un LSTM de même largeur (H4). Je compare résultats et coût avant de choisir.

Même couche unidirectionnelle, même dimension d'embedding, même largeur cachée, même couche de sortie que le LSTM. Tokenisation, vocabulaire, initialisation des embeddings et réglages restent communs ; seuls les poids diffèrent. Une largeur égale ne donne pas un nombre de paramètres égal : je les compte, et je compare F1 macro, dispersion, temps et écart train/validation, fold par fold face au LSTM.


In [58]:
# TODO : remplacer le LSTM par un GRU en conservant les autres choix de cette comparaison.
# TODO : vérifier les paramètres entraînables et le traitement du padding.
# TODO : conserver prédictions, courbes, epoch retenue et temps sur les mêmes folds.


> *Mes observations :*
>
> *Ma conclusion :*


## 4. Évaluation, erreurs et conclusions

### 4.1 Comparer la généralisation

Je rassemble accuracy et F1 macro par fold, puis moyenne et écart-type. La dispersion décrit cinq scores ; elle ne constitue pas un test de significativité.

Je compare les modèles sur les mêmes répliques. D'abord la différence de F1 macro fold par fold entre chaque réseau et LinearSVC, avec son signe. Puis, sur les 769 prédictions de validation, le nombre de répliques où un modèle a raison et l'autre tort. Je lis les courbes train/validation, jamais le score train seul, les métriques par classe, les paramètres et le temps total. Chaque matrice de confusion est commentée.


In [59]:
# TODO : afficher les scores par fold, les moyennes, les écarts-types, les paramètres et les temps.
# TODO : calculer les différences par fold avec LinearSVC et les comptes « A juste, B faux » sur les
#   769 prédictions.
# TODO : tracer et commenter les courbes train/validation enregistrées ; marquer l'epoch retenue.
# TODO : afficher précision, rappel, F1 et supports par classe, puis les matrices de confusion
#   commentées.


### 4.2 Revenir aux répliques

Je lis les erreurs communes et les désaccords entre ML et DL, en suivant JULIET–ROMEO et GLOUCESTER–DUKE VINCENTIO sans supposer que les erreurs du TP1 se reproduisent.

Le TP1 (section 8.3) suggérait que le modèle reconnaît la pièce plutôt que le style. Je le teste. Je regroupe les dix locuteurs par pièce à partir du corpus : les locuteurs qui se répondent dans une fenêtre de répliques consécutives appartiennent à la même pièce. Puis je compare la part d'erreurs intra-pièce à la part inter-pièce et au hasard. JULIET–ROMEO est intra-pièce ; GLOUCESTER–DUKE VINCENTIO est inter-pièce.

Quelles erreurs le DL corrige-t-il, lesquelles introduit-il ? Noms, vocabulaire partagé, longueur ou tokens inconnus donnent-ils une explication vérifiable ?


In [60]:
# TODO : afficher des répliques entières avec le label réel et les prédictions des modèles.
# TODO : reconstruire les pièces par voisinage de locuteurs dans le corpus ; mesurer erreurs intra-
#   pièce et inter-pièce.
# TODO : examiner cinq cas concrets, les confusions ciblées et les supports correspondants.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.3 Expériences ciblées, si les résultats le justifient

Un seul facteur change à la fois, une clé de `REGLAGES` ou une fonction de prétraitement, et je réentraîne dans les folds avant le test. Par priorité :

1. **Ordre des mots.** Mélanger les mots après tokenisation, avec un générateur fixé par l'indice de la réplique : ML et DL voient la même perturbation. Si H2 tient, LSTM et GRU baissent ; MLP et unigrammes TF–IDF ne bougent pas.
2. **Noms propres.** Masquer une liste de noms choisie sur le train du fold seulement. `kate` et `paulina` motivent la question depuis le TP1. Lecture par classe : les classes dont le F1 chute le plus sont reconnues par leurs noms.
3. **Régularisation.** Dropout et weight decay activés, contre le réglage de départ.
4. **Bidirectionnel.** LSTM ou GRU bidirectionnel, à largeur égale.
5. **Quantité de données.** 25, 50, 75 et 100 % du train de chaque fold pour LinearSVC et le meilleur réseau. La seule expérience qui dit si le deep learning manque de données ici.

Une comparaison MLP/LSTM/GRU ne suffit pas à attribuer un gain à l'ordre ; masquer quelques noms ne démontre pas que le signal restant est du style.


In [61]:
# TODO : choisir les expériences à mener sur les folds de développement, avant le test final.
# TODO : figer la perturbation ou la clé de REGLAGES modifiée, garder le reste du protocole et
#   comparer au modèle non modifié.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.4 Figer les choix et évaluer sur le test

Je choisis les configurations sur le F1 macro moyen en validation, avec leur dispersion et leur coût. Chaque réseau retenu prend son epoch retenue, reconstruit son vocabulaire sur les 769 répliques et se réentraîne depuis une nouvelle initialisation, sans utiliser le test pour s'arrêter. Une epoch sur 769 répliques compte 25 % de mises à jour de plus qu'une epoch de fold ; je le signale sans le corriger.

Les références ML sont réajustées sur les 769 répliques. Je compare les choix figés sur les 193 répliques de test, avec le repère du TP1, 0,563 de F1 macro pour LinearSVC, puis j'analyse les erreurs sans relancer de réglage.

Ce test a déjà servi au TP1 pour choisir LinearSVC : la comparaison reste exploratoire, et les hypothèses tirées de ses erreurs ne le rendent pas indépendant.


In [62]:
# TODO : figer les configurations et l'epoch retenue ; reconstruire vocabulaires et modèles sur les
#   769 répliques.
# TODO : évaluer les choix figés sur les 193 répliques réservées et conserver leurs prédictions.
# TODO : comparer scores, matrices de confusion commentées, erreurs et coût dans un tableau final.


> *Mes observations :*
>
> *Ma conclusion :*


### 4.5 Mon journal d'expériences

Une ligne après chaque essai ; une hypothèse reste même si le résultat ne la confirme pas.

| Expérience | Hypothèse | Changement | Observation chiffrée | Décision |
|---|---|---|---|---|
| 1. Tokenizer commun (section 3.3) | « I » et les contractions aident LinearSVC | `token_pattern` par défaut remplacé par `tokeniser` | | |

### 4.6 Ce que je peux conclure

Quatre questions : quel signal je prédis, ce que chaque modèle apprend réellement, sur quelles données il généralise, quelle métrique traduit le besoin. Quel modèle répond le mieux au problème, scores et coût compris ? Quelles hypothèses, de H1 à H4, mes expériences soutiennent-elles ? Qu'est-ce qui reste incertain sur le contexte, le style et la généralisation ?

> *Ma conclusion :*

### Pour aller plus loin

Prochaines pistes, hors de ce TP : un encodeur pré-entraîné de type BERT pour classer les répliques, et l'attention au niveau des caractères sur ce même corpus.
